# Chapter 11.8: Trustworthy Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Assess model robustness** by measuring sensitivity to adversarial inputs
2. **Simulate shilling attacks** (profile injection) on recommendation systems
3. **Detect data poisoning** attacks that manipulate recommendation rankings
4. **Implement defense mechanisms** including anomaly detection and robust aggregation
5. **Understand privacy attacks** (membership inference, attribute inference) on recommender models
6. **Design federated evaluation** protocols that evaluate without centralising user data
7. **Build a complete attack-and-defense pipeline** for recommendation security

## Prerequisites

- Chapters 11.1-11.7
- Basic understanding of collaborative filtering (matrix factorisation)
- Familiarity with anomaly detection concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part11/chapter_11.8_trustworthy.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part11/chapter_11.8_trustworthy.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import IsolationForest

np.random.seed(42)
plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. Threat Model for Recommendation Systems

| Attack Type | Goal | Method | Difficulty |
|-------------|------|--------|------------|
| **Shilling (Profile Injection)** | Promote/demote target items | Create fake user profiles | Medium |
| **Data Poisoning** | Corrupt training data | Inject fake interactions | Medium |
| **Model Extraction** | Steal the recommendation model | Query the API | Low |
| **Membership Inference** | Determine if a user is in training data | Observe model outputs | Low |
| **Attribute Inference** | Infer sensitive user attributes | Analyse recommendations | Medium |

Reference: Deldjoo et al., "A Survey on Adversarial Recommender Systems: From Attack/Defense Strategies to Generative Adversarial Networks" (ACM Computing Surveys, 2021).

> **💡 Concept:** Recommendation systems are particularly vulnerable because they are trained on user-generated data. Unlike image classifiers, an attacker can directly influence the training data by creating fake accounts.

In [ ]:
# Build a simple collaborative filtering system to attack
rng = np.random.RandomState(42)
NUM_USERS = 500
NUM_ITEMS = 200
LATENT_DIM = 10

# True latent factors
user_factors = rng.randn(NUM_USERS, LATENT_DIM) * 0.5
item_factors = rng.randn(NUM_ITEMS, LATENT_DIM) * 0.5

# Generate interaction matrix (ratings 1-5)
true_ratings = user_factors @ item_factors.T + 3.0  # center around 3
true_ratings = np.clip(true_ratings, 1, 5)

# Observe only a fraction of ratings
observe_prob = 0.05
observed_mask = rng.random((NUM_USERS, NUM_ITEMS)) < observe_prob
observed_ratings = np.where(observed_mask, np.round(true_ratings + rng.randn(NUM_USERS, NUM_ITEMS) * 0.5), 0)
observed_ratings = np.clip(observed_ratings, 0, 5)

# Simple recommendation: predict ratings using observed data and mean imputation
def simple_recommender(ratings_matrix, k=10):
    """Simple item-based collaborative filtering."""
    # Fill unobserved with item means
    filled = ratings_matrix.copy()
    for j in range(ratings_matrix.shape[1]):
        observed = ratings_matrix[:, j][ratings_matrix[:, j] > 0]
        if len(observed) > 0:
            mean_val = observed.mean()
        else:
            mean_val = 3.0
        filled[:, j] = np.where(ratings_matrix[:, j] > 0, ratings_matrix[:, j], mean_val)
    
    # SVD-based prediction
    U, S, Vt = np.linalg.svd(filled - filled.mean(axis=0), full_matrices=False)
    pred = U[:, :LATENT_DIM] @ np.diag(S[:LATENT_DIM]) @ Vt[:LATENT_DIM, :] + filled.mean(axis=0)
    
    # Generate top-k recommendations
    recommendations = {}
    for u in range(ratings_matrix.shape[0]):
        # Exclude already-rated items
        scores = pred[u].copy()
        scores[ratings_matrix[u] > 0] = -np.inf
        top_k = np.argsort(-scores)[:k]
        recommendations[u] = list(top_k)
    return recommendations, pred

clean_recs, clean_pred = simple_recommender(observed_ratings)
print(f"Clean system: {NUM_USERS} users, {NUM_ITEMS} items")
print(f"Observed ratings: {(observed_ratings > 0).sum()} / {NUM_USERS * NUM_ITEMS}")
print(f"Sparsity: {1 - (observed_ratings > 0).sum() / (NUM_USERS * NUM_ITEMS):.4f}")

## 2. Shilling Attacks

In a **shilling attack** (also called profile injection), the attacker creates fake user profiles that rate the target item highly and rate filler items to blend in.

### Common Attack Strategies

| Strategy | Filler Items | Target Item | Blend-In |
|----------|-------------|-------------|----------|
| **Random** | Random ratings | Max rating | Low |
| **Average** | Average ratings per item | Max rating | Medium |
| **Bandwagon** | Rate popular items highly | Max rating | High |

Reference: Lam & Riedl, "Shilling Recommender Systems for Fun and Profit" (WWW 2004).

In [ ]:
def generate_shilling_profiles(n_fake, num_items, target_item, attack_type='average',
                                filler_size=20, observed_ratings=None, rng=None):
    """Generate fake user profiles for a shilling attack."""
    if rng is None:
        rng = np.random.RandomState()
    
    fake_ratings = np.zeros((n_fake, num_items))
    
    for i in range(n_fake):
        # Select filler items (random subset, excluding target)
        candidates = [j for j in range(num_items) if j != target_item]
        filler_items = rng.choice(candidates, size=filler_size, replace=False)
        
        if attack_type == 'random':
            # Random ratings for fillers
            fake_ratings[i, filler_items] = rng.randint(1, 6, size=filler_size)
        
        elif attack_type == 'average':
            # Average rating per filler item
            for j in filler_items:
                item_ratings = observed_ratings[:, j][observed_ratings[:, j] > 0]
                if len(item_ratings) > 0:
                    fake_ratings[i, j] = np.round(item_ratings.mean())
                else:
                    fake_ratings[i, j] = 3
        
        elif attack_type == 'bandwagon':
            # Rate popular items highly
            item_counts = (observed_ratings > 0).sum(axis=0)
            popular = np.argsort(-item_counts)
            popular_fillers = [p for p in popular if p != target_item][:filler_size]
            fake_ratings[i, popular_fillers] = rng.choice([4, 5], size=len(popular_fillers))
        
        # Target item always gets max rating
        fake_ratings[i, target_item] = 5.0
    
    return fake_ratings

# Attack parameters
TARGET_ITEM = 150  # item we want to promote
N_FAKE = 50        # number of fake profiles

# Check target item's baseline ranking
target_rank_before = []
for u in range(NUM_USERS):
    scores = clean_pred[u].copy()
    scores[observed_ratings[u] > 0] = -np.inf
    rank = (scores > scores[TARGET_ITEM]).sum() + 1
    target_rank_before.append(rank)

print(f"Target item {TARGET_ITEM} before attack:")
print(f"  Average rank across users: {np.mean(target_rank_before):.1f}")
print(f"  Median rank: {np.median(target_rank_before):.1f}")
print(f"  In top-10 for {sum(r <= 10 for r in target_rank_before)} users")

In [ ]:
# Execute the attack
attack_results = {}

for attack_type in ['random', 'average', 'bandwagon']:
    fake_profiles = generate_shilling_profiles(
        N_FAKE, NUM_ITEMS, TARGET_ITEM, attack_type=attack_type,
        filler_size=20, observed_ratings=observed_ratings, rng=rng
    )
    
    # Inject fake profiles into the rating matrix
    poisoned_ratings = np.vstack([observed_ratings, fake_profiles])
    
    # Retrain recommender
    poisoned_recs, poisoned_pred = simple_recommender(poisoned_ratings)
    
    # Check target item's ranking after attack (only for real users)
    target_rank_after = []
    for u in range(NUM_USERS):
        scores = poisoned_pred[u].copy()
        scores[observed_ratings[u] > 0] = -np.inf
        rank = (scores > scores[TARGET_ITEM]).sum() + 1
        target_rank_after.append(rank)
    
    in_top10_before = sum(r <= 10 for r in target_rank_before)
    in_top10_after = sum(r <= 10 for r in target_rank_after)
    
    attack_results[attack_type] = {
        'avg_rank_before': np.mean(target_rank_before),
        'avg_rank_after': np.mean(target_rank_after),
        'top10_before': in_top10_before,
        'top10_after': in_top10_after,
    }
    
    print(f"\n{attack_type.upper()} attack ({N_FAKE} fake profiles):")
    print(f"  Avg rank: {np.mean(target_rank_before):.1f} -> {np.mean(target_rank_after):.1f}")
    print(f"  In top-10: {in_top10_before} -> {in_top10_after} users")

In [ ]:
# Visualise attack effectiveness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

attacks = list(attack_results.keys())
avg_ranks_before = [attack_results[a]['avg_rank_before'] for a in attacks]
avg_ranks_after = [attack_results[a]['avg_rank_after'] for a in attacks]
top10_before = [attack_results[a]['top10_before'] for a in attacks]
top10_after = [attack_results[a]['top10_after'] for a in attacks]

x = np.arange(len(attacks))
width = 0.3
axes[0].bar(x - width/2, avg_ranks_before, width, label='Before Attack', color='steelblue')
axes[0].bar(x + width/2, avg_ranks_after, width, label='After Attack', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels([a.capitalize() for a in attacks])
axes[0].set_ylabel('Average Rank (lower = more visible)')
axes[0].set_title('Attack Effectiveness: Target Item Rank')
axes[0].legend()
axes[0].invert_yaxis()

axes[1].bar(x - width/2, top10_before, width, label='Before Attack', color='steelblue')
axes[1].bar(x + width/2, top10_after, width, label='After Attack', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels([a.capitalize() for a in attacks])
axes[1].set_ylabel('# Users seeing target in top-10')
axes[1].set_title('Attack Effectiveness: Top-10 Exposure')
axes[1].legend()

plt.suptitle(f'Shilling Attack on Item {TARGET_ITEM} ({N_FAKE} fake profiles)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Detecting Shilling Attacks

Fake profiles exhibit statistical anomalies:
- **Unusual rating patterns**: too many high ratings, unusual rating distribution
- **Temporal anomalies**: burst of new accounts rating the same item
- **Feature-based**: rating deviation from mean, diversity of rated items

> **🔑 Pro Tip:** Combining multiple detection signals (statistical features + temporal patterns + network analysis) is much more effective than any single method.

In [ ]:
# Detection: extract features from user profiles
def extract_user_features(ratings_matrix):
    """Extract statistical features for each user profile."""
    n_users = ratings_matrix.shape[0]
    features = []
    
    for u in range(n_users):
        rated_items = ratings_matrix[u][ratings_matrix[u] > 0]
        n_rated = len(rated_items)
        
        if n_rated == 0:
            features.append([0, 0, 0, 0, 0, 0])
            continue
        
        # Feature 1: Number of ratings
        f1 = n_rated
        # Feature 2: Mean rating
        f2 = rated_items.mean()
        # Feature 3: Rating standard deviation
        f3 = rated_items.std() if n_rated > 1 else 0
        # Feature 4: Fraction of max ratings (5)
        f4 = (rated_items == 5).sum() / n_rated
        # Feature 5: Rating deviation from item means
        deviations = []
        for j in range(ratings_matrix.shape[1]):
            if ratings_matrix[u, j] > 0:
                item_mean = ratings_matrix[:, j][ratings_matrix[:, j] > 0].mean()
                deviations.append(abs(ratings_matrix[u, j] - item_mean))
        f5 = np.mean(deviations) if deviations else 0
        # Feature 6: DegSim - similarity to other suspicious users
        f6 = n_rated / ratings_matrix.shape[1]  # rating density
        
        features.append([f1, f2, f3, f4, f5, f6])
    
    return np.array(features)

# Use the bandwagon attack data (most sophisticated)
fake_profiles_bw = generate_shilling_profiles(
    N_FAKE, NUM_ITEMS, TARGET_ITEM, attack_type='bandwagon',
    filler_size=20, observed_ratings=observed_ratings, rng=rng
)
poisoned = np.vstack([observed_ratings, fake_profiles_bw])
labels = np.array([0] * NUM_USERS + [1] * N_FAKE)  # 0=genuine, 1=fake

# Extract features
user_feats = extract_user_features(poisoned)

# Use Isolation Forest for anomaly detection
iso_forest = IsolationForest(n_estimators=100, contamination=N_FAKE / (NUM_USERS + N_FAKE),
                              random_state=42)
iso_predictions = iso_forest.fit_predict(user_feats)
# -1 = anomaly (suspected fake), 1 = normal
detected_fake = (iso_predictions == -1).astype(int)

precision = precision_score(labels, detected_fake)
recall = recall_score(labels, detected_fake)
f1 = f1_score(labels, detected_fake)

print(f"=== Shilling Attack Detection ===")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"True fakes detected: {(detected_fake[NUM_USERS:] == 1).sum()} / {N_FAKE}")
print(f"False alarms on genuine: {(detected_fake[:NUM_USERS] == 1).sum()} / {NUM_USERS}")

In [ ]:
# Visualise the feature space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genuine_feats = user_feats[:NUM_USERS]
fake_feats = user_feats[NUM_USERS:]

axes[0].scatter(genuine_feats[:, 1], genuine_feats[:, 3], alpha=0.3, s=15,
                label='Genuine', color='steelblue')
axes[0].scatter(fake_feats[:, 1], fake_feats[:, 3], alpha=0.7, s=30,
                label='Fake', color='red', marker='x')
axes[0].set_xlabel('Mean Rating')
axes[0].set_ylabel('Fraction of Max Ratings')
axes[0].set_title('User Feature Space')
axes[0].legend()

axes[1].scatter(genuine_feats[:, 4], genuine_feats[:, 5], alpha=0.3, s=15,
                label='Genuine', color='steelblue')
axes[1].scatter(fake_feats[:, 4], fake_feats[:, 5], alpha=0.7, s=30,
                label='Fake', color='red', marker='x')
axes[1].set_xlabel('Rating Deviation from Item Mean')
axes[1].set_ylabel('Rating Density')
axes[1].set_title('Anomaly Detection Features')
axes[1].legend()

plt.suptitle('Detecting Fake Profiles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Defense Mechanisms

### 4.1 Robust Aggregation

Instead of using the mean, use robust statistics that are less affected by outliers:
- Trimmed mean (remove top/bottom $\alpha\%$ of ratings)
- Median
- Winsorized mean

### 4.2 Trust-Based Filtering

Weight users by a trust score based on account age, rating history consistency, etc.

In [ ]:
def robust_recommender(ratings_matrix, method='trimmed_mean', trim_pct=0.1, k=10):
    """Recommender with robust aggregation."""
    filled = ratings_matrix.copy()
    
    for j in range(ratings_matrix.shape[1]):
        observed = ratings_matrix[:, j][ratings_matrix[:, j] > 0]
        if len(observed) > 0:
            if method == 'trimmed_mean':
                mean_val = stats.trim_mean(observed, trim_pct)
            elif method == 'median':
                mean_val = np.median(observed)
            else:  # standard mean
                mean_val = observed.mean()
        else:
            mean_val = 3.0
        filled[:, j] = np.where(ratings_matrix[:, j] > 0, ratings_matrix[:, j], mean_val)
    
    U, S, Vt = np.linalg.svd(filled - filled.mean(axis=0), full_matrices=False)
    pred = U[:, :LATENT_DIM] @ np.diag(S[:LATENT_DIM]) @ Vt[:LATENT_DIM, :] + filled.mean(axis=0)
    
    return pred

# Compare defense effectiveness
poisoned_bw = np.vstack([observed_ratings, fake_profiles_bw])

for method in ['mean', 'trimmed_mean', 'median']:
    pred = robust_recommender(poisoned_bw, method=method)
    
    # Check target item rank for genuine users
    target_ranks = []
    for u in range(NUM_USERS):
        scores = pred[u].copy()
        scores[observed_ratings[u] > 0] = -np.inf
        rank = (scores > scores[TARGET_ITEM]).sum() + 1
        target_ranks.append(rank)
    
    in_top10 = sum(r <= 10 for r in target_ranks)
    print(f"{method:15s}  Avg rank: {np.mean(target_ranks):.1f}, In top-10: {in_top10} users")

print(f"{'No attack':15s}  Avg rank: {np.mean(target_rank_before):.1f}, In top-10: {sum(r <= 10 for r in target_rank_before)} users")

## 5. Privacy Attacks on Recommendations

### Membership Inference

Can an adversary determine whether a specific user's data was used to train the model?

### Attribute Inference

Can an adversary infer sensitive attributes (gender, age, political leaning) from a user's recommendations?

Reference: Zhang et al., "Membership Inference Attacks Against Recommender Systems" (CCS 2021).

> **⚠️ Common Pitfall:** Even "anonymised" recommendation data can leak private information. Netflix Prize data was famously de-anonymised by linking with IMDb ratings.

In [ ]:
# Simulate membership inference attack
# Idea: members (in training set) will have lower prediction error
# than non-members

# Create member and non-member sets
n_members = 200
n_non_members = 200
member_users = rng.choice(NUM_USERS, size=n_members, replace=False)

# Non-members: generate new user profiles NOT in the training data
non_member_factors = rng.randn(n_non_members, LATENT_DIM) * 0.5
non_member_ratings = non_member_factors @ item_factors.T + 3.0
non_member_ratings = np.clip(non_member_ratings, 1, 5)

# For each user, compute the prediction error on their known ratings
member_errors = []
for u in member_users:
    rated = observed_ratings[u] > 0
    if rated.sum() > 0:
        error = np.mean((clean_pred[u][rated] - observed_ratings[u][rated]) ** 2)
        member_errors.append(error)

non_member_errors = []
for u in range(n_non_members):
    # Select some items and compute prediction error
    test_items = rng.choice(NUM_ITEMS, size=10, replace=False)
    error = np.mean((clean_pred[0][test_items] - non_member_ratings[u][test_items]) ** 2)
    non_member_errors.append(error)

# Attack: classify based on threshold
threshold = np.median(member_errors + non_member_errors)
member_detected = sum(1 for e in member_errors if e < threshold)
non_member_detected = sum(1 for e in non_member_errors if e >= threshold)

accuracy = (member_detected + non_member_detected) / (n_members + n_non_members)

print(f"Membership Inference Attack:")
print(f"  Members correctly identified: {member_detected}/{len(member_errors)}")
print(f"  Non-members correctly identified: {non_member_detected}/{len(non_member_errors)}")
print(f"  Overall accuracy: {accuracy:.4f} (random = 0.50)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(member_errors, bins=30, alpha=0.6, label='Members', color='steelblue', density=True)
ax.hist(non_member_errors, bins=30, alpha=0.6, label='Non-members', color='coral', density=True)
ax.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold={threshold:.2f}')
ax.set_xlabel('Prediction Error (MSE)')
ax.set_ylabel('Density')
ax.set_title('Membership Inference: Error Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Federated Evaluation

Federated evaluation allows evaluating recommendation models without centralising user data:

1. Each user device computes local metrics on private data
2. Only aggregated statistics are shared with the server
3. Differential privacy noise can be added for extra protection

Reference: Ammad-ud-din et al., "Federated Collaborative Filtering for On-Device Next App Prediction" (2019, Google).

In [ ]:
class FederatedEvaluator:
    """Simulate federated evaluation of recommendations."""
    
    def __init__(self, epsilon=1.0):
        self.epsilon = epsilon  # differential privacy parameter
    
    def local_evaluate(self, user_recs, user_truth, k=10):
        """Compute metrics locally on one user's device."""
        # Hit rate
        hit = 1.0 if len(set(user_recs[:k]) & user_truth) > 0 else 0.0
        
        # NDCG
        rels = [1.0 if item in user_truth else 0.0 for item in user_recs[:k]]
        dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rels))
        n_rel = min(len(user_truth), k)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        
        return {'hit': hit, 'ndcg': ndcg}
    
    def add_noise(self, value, sensitivity=1.0):
        """Add Laplacian noise for differential privacy."""
        scale = sensitivity / self.epsilon
        return value + np.random.laplace(0, scale)
    
    def federated_aggregate(self, local_metrics, private=True):
        """Aggregate local metrics with optional DP noise."""
        n_users = len(local_metrics)
        results = {}
        
        for metric_name in local_metrics[0].keys():
            values = [m[metric_name] for m in local_metrics]
            raw_mean = np.mean(values)
            
            if private:
                # Each user adds noise to their local metric before sharing
                noisy_values = [self.add_noise(v, sensitivity=1.0/n_users) for v in values]
                private_mean = np.mean(noisy_values)
                results[metric_name] = {
                    'raw': raw_mean,
                    'private': private_mean,
                    'noise_std': abs(private_mean - raw_mean)
                }
            else:
                results[metric_name] = {'raw': raw_mean}
        
        return results

# Simulate federated evaluation
# Create ground truth for evaluation
eval_ground_truth = {}
for u in range(NUM_USERS):
    # Top items by true rating that were not in training
    unobserved = np.where(observed_ratings[u] == 0)[0]
    if len(unobserved) > 0:
        top_unobserved = unobserved[np.argsort(-true_ratings[u][unobserved])[:5]]
        eval_ground_truth[u] = set(top_unobserved)

fed_eval = FederatedEvaluator(epsilon=1.0)

# Collect local metrics
local_results = []
for u in range(NUM_USERS):
    if u in eval_ground_truth and u in clean_recs:
        local_results.append(fed_eval.local_evaluate(clean_recs[u], eval_ground_truth[u]))

# Aggregate with different privacy levels
print("=== Federated Evaluation Results ===")
for eps in [0.1, 0.5, 1.0, 5.0, float('inf')]:
    fed_eval.epsilon = eps
    agg = fed_eval.federated_aggregate(local_results, private=(eps != float('inf')))
    eps_str = f"eps={eps:.1f}" if eps != float('inf') else "no privacy"
    print(f"  {eps_str:15s} | NDCG: {agg['ndcg']['raw' if eps == float('inf') else 'private']:.4f} "
          f"| HitRate: {agg['hit']['raw' if eps == float('inf') else 'private']:.4f}")

## 7. Exercises

### Exercise 1: Simulate and Defend Against a Shilling Attack

In [ ]:
# 🏋️ Exercise 1: Full Attack-Defense Pipeline
#
# TODO:
# 1. Implement a "segment" attack: fake profiles only rate items in
#    the target item's category (more targeted than bandwagon)
# 2. Apply the Isolation Forest detector — does it catch this attack?
# 3. Implement a defense: remove detected fake profiles, then retrain
# 4. Compare target item rank: before attack, after attack, after defense
# 5. Plot all three states in a bar chart

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

### Exercise 2: Attack Scalability Analysis

In [ ]:
# 🏋️ Exercise 2: Attack Scalability
#
# TODO: Investigate how the number of fake profiles affects attack success.
# 1. Vary N_FAKE from 5 to 200 in steps
# 2. For each N_FAKE, run the bandwagon attack and measure:
#    a. Average target item rank change
#    b. Detection rate (using Isolation Forest)
# 3. Plot: x = number of fake profiles, y1 = rank change, y2 = detection rate
# 4. At what point does the detection rate make the attack impractical?

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

### Exercise 3: Privacy-Utility Trade-off

In [ ]:
# 🏋️ Exercise 3: Privacy-Utility Trade-off in Federated Evaluation
#
# TODO:
# 1. Run federated evaluation with epsilon values from 0.01 to 100
# 2. For each epsilon, run 50 trials to get the expected noisy metric
#    and its standard deviation
# 3. Plot: x = epsilon, y = NDCG@10 estimate with error bars
# 4. At what epsilon does the noise become negligible (<1% error)?
# 5. Discuss: is this epsilon value sufficient for meaningful privacy?

# --- YOUR CODE HERE ---

# --- END YOUR CODE ---

## Summary

In this notebook we covered:

- **Shilling attacks**: fake profile injection to manipulate recommendations; random, average, and bandwagon strategies
- **Attack detection**: statistical feature extraction + Isolation Forest for anomaly detection
- **Defense mechanisms**: robust aggregation (trimmed mean, median), trust-based filtering
- **Privacy attacks**: membership inference exploiting prediction confidence differences
- **Federated evaluation**: evaluating models without centralising data; differential privacy trade-offs

**Key takeaway:** Trustworthy recommendation systems must defend against both data integrity threats (shilling, poisoning) and privacy threats (inference attacks). Building in detection, robust aggregation, and privacy protection from the start is essential for real-world deployment.